# Oxford-IIIT Pets: kích thước ảnh và tỉ lệ lớp của trimap

Ảnh lấy từ mirror HF `oxford-pets` (parquet), trimap từ `oxford-pets-trimaps` (PNG, giá trị 1 thú cưng, 2 nền, 3 viền). Điền các hàm trong cell chấm điểm; cell `explore` chỉ để xem. Trước khi viết service phân đoạn, bạn cần biết ảnh thật lớn cỡ nào (để đặt giới hạn kích thước) và lớp nào hiếm (để chọn metric).

In [ ]:
# explore: đưa gốc repo vào sys.path và tải dữ liệu nếu chưa có
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

ensure("oxford-pets")
ensure("oxford-pets-trimaps")
PETS = ROOT / "data" / "raw" / "oxford-pets"
TRIMAPS = ROOT / "data" / "raw" / "oxford-pets-trimaps" / "annotations" / "trimaps"


In [ ]:
import io
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from PIL import Image

TRIMAP_CLASSES = {1: "pet", 2: "background", 3: "border"}
SPLITS = ("train", "test")

In [ ]:
def list_trimaps(trimap_dir):
    return sorted(p for p in Path(trimap_dir).glob("*.png") if not p.name.startswith("._"))

In [ ]:
def trimap_profile(paths):
    rows = []
    for path in paths:
        values = np.asarray(Image.open(path))
        counts = np.bincount(values.ravel(), minlength=4)
        row = {"image_id": Path(path).stem, "height": values.shape[0], "width": values.shape[1]}
        row.update({name: counts[value] / values.size for value, name in TRIMAP_CLASSES.items()})
        rows.append(row)
    return pd.DataFrame(rows, columns=["image_id", "height", "width", *TRIMAP_CLASSES.values()])

In [ ]:
def image_table(pets_dir):
    rows = []
    for split in SPLITS:
        table = pq.read_table(Path(pets_dir) / "data" / f"{split}-00000-of-00001.parquet", columns=["image", "image_id"])
        for image, image_id in zip(table.column("image").to_pylist(), table.column("image_id").to_pylist()):
            with Image.open(io.BytesIO(image["bytes"])) as img:
                width, height = img.size
                rows.append({"image_id": image_id, "split": split, "height": height, "width": width, "mode": img.mode})
    return pd.DataFrame(rows, columns=["image_id", "split", "height", "width", "mode"])

In [ ]:
def join_report(trimaps, images):
    merged = trimaps.merge(images, on="image_id", suffixes=("_trimap", "_image"))
    mismatch = (merged["height_trimap"] != merged["height_image"]) | (merged["width_trimap"] != merged["width_image"])
    return {
        "matched": int(len(merged)),
        "trimap_only": int(len(set(trimaps["image_id"]) - set(images["image_id"]))),
        "image_only": int(len(set(images["image_id"]) - set(trimaps["image_id"]))),
        "size_mismatch": int(mismatch.sum()),
    }

In [ ]:
def share_over(images, max_side):
    return float((images[["height", "width"]].max(axis=1) > max_side).mean())

In [ ]:
def without_pet(trimaps):
    return sorted(trimaps.loc[trimaps["pet"] == 0, "image_id"])

In [ ]:
# explore: hồ sơ trimap và ảnh
trimap_paths = list_trimaps(TRIMAPS)
print("trimap hợp lệ:", len(trimap_paths), "| mọi tệp *.png:", len(list(TRIMAPS.glob("*.png"))))
trimaps = trimap_profile(trimap_paths)
images = image_table(PETS)
display(trimaps.describe().round(3))
display(images["mode"].value_counts())
print(join_report(trimaps, images))

In [ ]:
# explore: kích thước ảnh và quyết định giới hạn của service
side = images[["height", "width"]].max(axis=1)
print(side.quantile([0.5, 0.9, 0.99, 1.0]))
for limit in (500, 800, 1024, 2048):
    print(f"cạnh dài > {limit}: {share_over(images, limit):.2%}")
print("ảnh có cạnh ngắn < 128:", int((images[["height", "width"]].min(axis=1) < 128).sum()))

In [ ]:
# explore: tỉ lệ lớp và trimap không có thú cưng
print(trimaps[list(TRIMAP_CLASSES.values())].mean().round(4))
empty = without_pet(trimaps)
print(len(empty), "trimap không có pixel thú cưng, ví dụ:", empty[:5])

In [ ]:
# explore: biểu đồ
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
side.clip(upper=1200).plot.hist(bins=60, ax=axes[0], title="Cạnh dài nhất (px, chặn ở 1200)")
trimaps[list(TRIMAP_CLASSES.values())].plot.box(ax=axes[1], title="Tỉ lệ pixel mỗi lớp trên một ảnh")
fig.tight_layout()
fig.savefig("pets_profile.png", dpi=110)